# CodeLens AI — QLoRA Fine-Tuning Notebook
**Course:** CSC-233 Artificial Intelligence Lab | **Base model:** google/gemma-3-4b-it

### Before running:
1. Set runtime to **GPU (T4)** → Runtime > Change runtime type > T4
2. Accept Gemma 3 terms at [huggingface.co/google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it)
3. Get your HF token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
4. Fill in the **CONFIG cell** below with your HF token
5. Click **Runtime > Run all**

---

## 0 — GPU Check & Install

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to: Settings > Accelerator > GPU T4 x2'
    )

N_GPUS   = torch.cuda.device_count()
gpu_info = [torch.cuda.get_device_name(i) for i in range(N_GPUS)]
vram_gb  = sum(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPUS)) / 1e9

print(f'GPUs : {N_GPUS}x {gpu_info[0]}')
print(f'VRAM : {vram_gb:.1f} GB total')

# T4 does NOT support bfloat16 — auto-detect
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_BF16      = torch.cuda.is_bf16_supported()
DTYPE_NAME    = 'bfloat16' if USE_BF16 else 'float16'
print(f'Dtype: {DTYPE_NAME} (auto-detected)')
print(f'\nReady. {N_GPUS} GPU(s) will be used for training.')

In [ ]:
import subprocess, sys

# Pin ranges to avoid breaking API changes between major versions
pkgs = [
    'transformers>=4.47.0,<4.51.0',
    'peft>=0.14.0,<0.16.0',
    'trl>=0.12.0,<0.13.0',
    'datasets>=2.20.0',
    'bitsandbytes>=0.45.0',
    'accelerate>=1.2.0',
    'sentencepiece',
    'protobuf',
    'huggingface_hub>=0.26.0',
    'numpy',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

# Verify all critical imports load correctly
import importlib
ok = True
for pkg in ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg:<15} {m.__version__}')
    except ImportError as e:
        print(f'  {pkg:<15} MISSING — {e}')
        ok = False

if not ok:
    raise RuntimeError('Some packages failed. Go to Runtime > Restart session, then run all cells again.')
print('\nAll packages ready.')

## 1 — Configuration
> **Edit this cell.** Everything else runs automatically.

In [ ]:
import os

# ── EDIT THESE (optional) ───────────────────────────────────────
HF_TOKEN    = ''   # NOT required — Qwen2.5-Coder is open access
# ───────────────────────────────────────────────────────────────

BASE_MODEL  = 'Qwen/Qwen2.5-Coder-3B-Instruct'

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    # Dataset added as "codelens-dataset" in notebook settings
    TRAIN_FILE = '/kaggle/input/datasets/mehkaankhan/codelens-dataset-1/train.jsonl'
    VAL_FILE   = '/kaggle/input/datasets/mehkaankhan/codelens-dataset-1/val.jsonl'
    OUTPUT_DIR = '/kaggle/working/adapter'
    MERGED_DIR = '/kaggle/working/merged'
    GGUF_PATH  = '/kaggle/working/codelens-qwen-q4_k_m.gguf'
else:
    TRAIN_FILE = '/content/train.jsonl'
    VAL_FILE   = '/content/val.jsonl'
    OUTPUT_DIR = '/content/drive/MyDrive/codelens/adapter'
    MERGED_DIR = '/content/drive/MyDrive/codelens/merged'
    GGUF_PATH  = '/content/drive/MyDrive/codelens/codelens-qwen-q4_k_m.gguf'

# LoRA
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

# Training — 1 epoch on 63K examples fits well within Kaggle 12h limit
NUM_EPOCHS    = 1
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
LR            = 2e-4
MAX_SEQ_LEN   = 1024
WARMUP_STEPS  = 100
LOGGING_STEPS = 10
EVAL_STEPS    = 200
SAVE_STEPS    = 200

print(f'Environment : {"Kaggle" if IS_KAGGLE else "Colab"}')
print(f'Base model  : {BASE_MODEL}')
print(f'Train file  : {TRAIN_FILE}')
print(f'Output dir  : {OUTPUT_DIR}')
print('Config loaded.')


## 1b — HuggingFace Login

**Two ways to authenticate (pick one):**
- **Recommended:** Add `HF_TOKEN` to Colab Secrets → click the 🔑 icon in the left sidebar → **+ Add new secret** → Name: `HF_TOKEN`, Value: your token. Reused across sessions, never visible in the notebook.
- **Fallback:** Paste your token directly in the config cell above.

**Also required before this will work:** Accept Gemma 3 terms at [huggingface.co/google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) — without this you get a 401 even with a valid token.

In [ ]:
from huggingface_hub import login
import os

token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        print('Using HF token from Kaggle Secrets.')
except Exception:
    pass

# Try Colab secrets
if not token:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            print('Using HF token from Colab Secrets.')
    except Exception:
        pass

# Try config cell variable
if not token and HF_TOKEN:
    token = HF_TOKEN
    print('Using HF token from config cell.')

# Login only if a token was found — Qwen2.5-Coder does NOT require one
if token:
    login(token=token)
    HF_TOKEN = token
    print('Logged in to HuggingFace Hub.')
else:
    print('No HF token found — skipping login.')
    print('This is fine: Qwen2.5-Coder-3B-Instruct is open access.')

## 2 — Dataset

In [ ]:
import os, subprocess, sys

# Detect environment and mount Drive accordingly
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    # On Kaggle, install and mount Google Drive via google-colab package
    subprocess.run(['pip', 'install', '-q', 'google-colab'], check=False)
    try:
        from googleapiclient.discovery import build
        print('Kaggle environment detected.')
        print('Outputs will save to /kaggle/working/ and be downloadable from the Output tab.')
    except Exception:
        print('Kaggle environment — outputs go to /kaggle/working/')

elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/codelens', exist_ok=True)
    print('Colab + Drive mounted.')

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Colab"}')

In [ ]:
import os

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    # Files already mounted from the dataset — no upload needed
    missing = [p for p in [TRAIN_FILE, VAL_FILE] if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            f"Dataset files not found: {missing}\n"
            "Fix: Notebook Settings > Add data > Your datasets > codelens-dataset"
        )
    print(f'train.jsonl : OK  ({os.path.getsize(TRAIN_FILE)//1024//1024} MB)')
    print(f'val.jsonl   : OK  ({os.path.getsize(VAL_FILE)//1024//1024} MB)')
else:
    from google.colab import files

    def upload_if_missing(path, label):
        if os.path.exists(path):
            print(f'Found: {path}')
            return
        print(f'Upload {label} ({os.path.basename(path)}):')
        uploaded = files.upload()
        for name, content in uploaded.items():
            dest = f'/content/{name}'
            with open(dest, 'wb') as fh:
                fh.write(content)
            print(f'Saved to {dest}')

    upload_if_missing(TRAIN_FILE, 'training data')
    upload_if_missing(VAL_FILE,   'validation data')
    print(f'train.jsonl : {os.path.exists(TRAIN_FILE)}')
    print(f'val.jsonl   : {os.path.exists(VAL_FILE)}')


In [ ]:
from datasets import load_dataset

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train[':'15000']')
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train')

print(f'Train : {len(train_ds):,} samples')
print(f'Val   : {len(val_ds):,} samples')
print(f'Fields: {list(train_ds.features.keys())}')

# Validate expected fields exist
required = {'system_prompt', 'user_input', 'assistant_output'}
actual   = set(train_ds.features.keys())
if not required.issubset(actual):
    raise ValueError(
        f'Missing fields: {required - actual}\n'
        f'Your dataset has: {actual}\n'
        'Update format_sample() below to match your field names.'
    )
print('\nField validation passed.')

## 3 — Tokenizer & Dataset Formatting

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Pad token  : {tokenizer.pad_token}')

In [ ]:
def format_sample(sample):
    messages = [
        {
            'role': 'user',
            'content': f"[SYSTEM]: {sample['system_prompt']}\n\n{sample['user_input']}"
        },
        {
            'role': 'assistant',
            'content': sample['assistant_output']
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

train_ds = train_ds.map(format_sample, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(format_sample,   remove_columns=val_ds.column_names)

print('Dataset formatted. Preview (first 600 chars):')
print('-' * 60)
print(train_ds[0]['text'][:600])
print('-' * 60)

# Token length stats
import numpy as np
lengths = [len(tokenizer(x['text'])['input_ids']) for x in train_ds.select(range(min(500, len(train_ds))))]
print(f'\nToken lengths (sample of 500):')
print(f'  Mean : {np.mean(lengths):.0f}')
print(f'  Max  : {np.max(lengths)}')
print(f'  p95  : {np.percentile(lengths, 95):.0f}')
if np.percentile(lengths, 95) > MAX_SEQ_LEN:
    print(f'\nTip: 95th percentile exceeds MAX_SEQ_LEN={MAX_SEQ_LEN}. Consider increasing it.')

## 4 — Load Model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Qwen2.5-Coder is open access — no HF token required
token_arg = {'token': HF_TOKEN} if HF_TOKEN else {}

print(f'Loading {BASE_MODEL} in 4-bit ({DTYPE_NAME})... (1-2 min)')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},  # single GPU — 4-bit + multi-GPU is broken
    torch_dtype=COMPUTE_DTYPE,
    attn_implementation='sdpa',   # Qwen supports scaled dot-product attention
    **token_arg,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Model loaded. Parameters: {total_params:.1f}B')

## 5 — Apply LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model

# prepare_model_for_kbit_training was deprecated in peft 0.14 — handle both
try:
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)
except Exception:
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable / 1e6:.1f}M')
print(f'Total params     : {total / 1e9:.1f}B')
print(f'Trainable %      : {100 * trainable / total:.2f}%')

## 6 — Train
> This cell takes **1-4 hours** depending on dataset size. Loss should decrease steadily.
> If val loss starts rising while train loss falls, the model is overfitting — stop early.

In [ ]:
import os, gc, torch
from trl import SFTTrainer, SFTConfig

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Scale batch size across all available GPUs
effective_batch = BATCH_SIZE * N_GPUS
grad_accum      = max(1, GRAD_ACCUM // N_GPUS)   # keep effective batch constant
print(f'GPUs            : {N_GPUS}')
print(f'Batch per GPU   : {BATCH_SIZE}')
print(f'Effective batch : {effective_batch}')
print(f'Grad accum      : {grad_accum}')

# Resume from checkpoint if one exists
resume_path = None
if os.path.exists(os.path.join(OUTPUT_DIR, 'trainer_state.json')):
    resume_path = OUTPUT_DIR
    print(f'\nCheckpoint found — resuming from {OUTPUT_DIR}')

def build_trainer(batch_size, grad_accum):
    config = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=grad_accum,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        eval_strategy='steps',
        eval_steps=EVAL_STEPS,
        save_strategy='steps',
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        bf16=USE_BF16,
        fp16=not USE_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        report_to='none',
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        packing=True,                        # pack sequences → better GPU utilisation → 2-3x faster
        dataloader_pin_memory=False,         # required for stable multi-GPU
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
    )
    return SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=config,
    )

# Auto-reduce batch on OOM: 4 → 2 → 1
batch_size   = BATCH_SIZE
g_accum      = grad_accum
train_result = None

for attempt in range(3):
    try:
        print(f'\nAttempt {attempt+1}: batch_size={batch_size} x {N_GPUS} GPUs, grad_accum={g_accum}')
        trainer      = build_trainer(batch_size, g_accum)
        train_result = trainer.train(resume_from_checkpoint=resume_path)
        break
    except torch.cuda.OutOfMemoryError:
        print(f'OOM at batch_size={batch_size}. Halving...')
        gc.collect()
        torch.cuda.empty_cache()
        batch_size = max(1, batch_size // 2)
        g_accum    = g_accum * 2
        if attempt == 2:
            raise RuntimeError(
                'OOM even at batch_size=1. Set MAX_SEQ_LEN=512 in the config cell and retry.'
            )

print('\nTraining complete!')
print(f'  Train loss  : {train_result.training_loss:.4f}')
print(f'  Total steps : {train_result.global_step}')

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\nAdapter saved to: {OUTPUT_DIR}')

## 7 — Merge LoRA Weights into Base Model
> Loads the base model fresh in float16 (not quantized) and merges the adapter weights.

In [ ]:
import os, gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

del model
gc.collect()
torch.cuda.empty_cache()

token_arg = {'token': HF_TOKEN} if HF_TOKEN else {}

print(f'Loading {BASE_MODEL} for merge (CPU, float16)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cpu',
    **token_arg,
)

print('Loading adapter...')
base_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

print('Merging...')
base_model = base_model.merge_and_unload()

print(f'Saving merged model to {MERGED_DIR}...')
os.makedirs(MERGED_DIR, exist_ok=True)
base_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print('Merge complete.')

## 8 — Quick Inference Test
> Sanity-check that the merged model responds correctly before exporting.

In [ ]:
from transformers import pipeline
import gc, torch

test_cases = [
    {
        'label': 'Summarization',
        'prompt': '[SYSTEM]: You are a code summarizer. Explain what this code does in plain English.\n\ndef fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)'
    },
    {
        'label': 'Translation',
        'prompt': '[SYSTEM]: You are a code translator. Convert the following Python code to Rust.\n\ndef add(a, b):\n    return a + b'
    },
]

pipe = pipeline(
    'text-generation',
    model=base_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    device_map={'': 0},  # single GPU — 4-bit + multi-GPU is broken
)

for tc in test_cases:
    print(f'=== {tc["label"]} ===')
    result = pipe(tc['prompt'])[0]['generated_text']
    response = result[len(tc['prompt']):].strip()
    print(response[:500])
    print()

del pipe
gc.collect()
torch.cuda.empty_cache()
print('Test complete.')

## 9 — Export to GGUF (for Ollama)
> Converts the merged HuggingFace model to GGUF format, then quantizes to Q4_K_M.
> Final file size: ~2.5 GB. Saved directly to your Google Drive.

In [ ]:
import os, subprocess, gc, torch, shutil, zipfile, io, stat

del base_model
gc.collect()
torch.cuda.empty_cache()

LLAMA_DIR = '/kaggle/working/llama.cpp' if IS_KAGGLE else '/content/llama.cpp'
F16_PATH  = ('/kaggle/working/codelens-qwen-f16.gguf' if IS_KAGGLE
             else '/content/codelens-qwen-f16.gguf')

# ── Step 1: Clone llama.cpp (Python scripts only — no build needed) ─────────
if not os.path.exists(LLAMA_DIR):
    print('Cloning llama.cpp...')
    subprocess.run(['git', 'clone', 'https://github.com/ggerganov/llama.cpp',
                    '--depth=1', LLAMA_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-r', f'{LLAMA_DIR}/requirements.txt'], check=True)
print('llama.cpp Python scripts ready.')

# ── Step 2: Convert merged HF model → GGUF f16 (no binary needed) ──────────
print('Converting to GGUF f16... (5-10 min)')
subprocess.run([
    'python', f'{LLAMA_DIR}/convert_hf_to_gguf.py',
    MERGED_DIR, '--outfile', F16_PATH, '--outtype', 'f16',
], check=True)
print(f'F16 GGUF: {os.path.getsize(F16_PATH) / 1e9:.1f} GB')

# ── Step 3: Download pre-built llama-quantize from GitHub Releases ───────────
QUANTIZE_BIN = None
try:
    import requests as _req
    print('Fetching latest llama.cpp release...')
    rel = _req.get(
        'https://api.github.com/repos/ggerganov/llama.cpp/releases/latest',
        timeout=30
    ).json()
    asset_url = None
    for asset in rel.get('assets', []):
        if 'ubuntu-x64' in asset['name'] and asset['name'].endswith('.zip'):
            asset_url = asset['browser_download_url']
            print(f'Downloading: {asset["name"]}')
            break
    if asset_url:
        r = _req.get(asset_url, timeout=180)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        qname = next(
            (n for n in z.namelist() if 'llama-quantize' in n and not n.endswith('/')),
            None
        )
        if qname:
            bin_path = '/kaggle/working/llama-quantize' if IS_KAGGLE else '/content/llama-quantize'
            with open(bin_path, 'wb') as bf:
                bf.write(z.read(qname))
            os.chmod(bin_path, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP)
            QUANTIZE_BIN = bin_path
            print(f'Quantize binary ready.')
except Exception as e:
    print(f'Could not get quantize binary: {e}')
    print('Will save as f16 GGUF instead (larger but works with Ollama).')

# ── Step 4: Quantize to Q4_K_M (or keep f16 as fallback) ────────────────────
if QUANTIZE_BIN and os.path.exists(QUANTIZE_BIN):
    print('Quantizing to Q4_K_M...')
    subprocess.run([QUANTIZE_BIN, F16_PATH, GGUF_PATH, 'Q4_K_M'], check=True)
    os.remove(F16_PATH)
    print(f'Q4_K_M GGUF saved: {GGUF_PATH}')
else:
    shutil.move(F16_PATH, GGUF_PATH)
    print(f'Saved as f16 GGUF: {GGUF_PATH}')
    print('Note: ~6 GB instead of ~2 GB, but Ollama loads it fine.')

print(f'\nFinal size: {os.path.getsize(GGUF_PATH) / 1e9:.2f} GB')


## 10 — Generate Modelfile & Local Setup Instructions
> Run this cell to create the Ollama Modelfile. Then follow the printed steps on your local machine.

In [ ]:
import os

MODELFILE_PATH = (
    '/content/drive/MyDrive/codelens/Modelfile'
    if not IS_KAGGLE else
    '/kaggle/working/Modelfile'
)

modelfile_content = '''FROM ./codelens-qwen-q4_k_m.gguf

SYSTEM """You are CodeLens AI, an expert programming assistant specialized in code understanding and cross-language translation. You help developers understand unfamiliar codebases by providing clear, accurate explanations. When translating code between languages, preserve the intent and semantics, not just the syntax. Be concise and educational."""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
'''

with open(MODELFILE_PATH, 'w') as f:
    f.write(modelfile_content)

print('Modelfile saved.')
print()
print('=' * 65)
print('NEXT STEPS on your LOCAL machine:')
print('=' * 65)
print()
print('1. Download from Output tab (Kaggle) or Drive (Colab):')
print('      codelens-qwen-q4_k_m.gguf  (~1.9 GB)')
print('      Modelfile')
print()
print('2. cd to the download folder and register with Ollama:')
print('      ollama create codelens-qwen -f Modelfile')
print()
print('3. Test it:')
print('      ollama run codelens-qwen')
print('      >>> Summarize this: def f(x): return x * 2')
print()
print('4. Confirm:')
print('      ollama list')
print('=' * 65)

---
## Done!

Your fine-tuned model is in Google Drive at `My Drive/codelens/`. 
Download both files and follow the instructions printed above to register with Ollama.

**Training artifacts saved:**
| File | Location | Purpose |
|---|---|---|
| `adapter/` | Drive/codelens/ | LoRA weights (checkpoint, ~80 MB) |
| `merged/` | Drive/codelens/ | Full merged model in HF format |
| `codelens-gemma-q4_k_m.gguf` | Drive/codelens/ | Ollama-ready quantized model |
| `Modelfile` | Drive/codelens/ | Ollama registration config |